# Prédiction de la consommation électrique — RNN/LSTM/GRU + MLP hybride

## Objectif

Ce notebook combine deux entrées de nature différente pour prédire la consommation électrique
des bâtiments **tout-électriques**  :

- une **entrée séquentielle** (consommation passée + météo horaire/journalière), encodée par un
  réseau récurrent (RNN simple, LSTM ou GRU — paramétrable, GRU par défaut) ;
- une **entrée statique** (métadonnées physiques du bâtiment : enveloppe thermique, occupation,
  gains solaires...), encodée par un MLP.

Les deux représentations sont concaténées puis passées dans une tête de régression commune.
C'est l'architecture dite *hybride séquence + tabulaire*.

Deux parties de prédiction sont traitées avec la même architecture :

1. **Partie A — Prévision à court terme (t+1)** : à partir d'une fenêtre glissante de 24 heures
   (consommation + météo), prédire la consommation de l'heure suivante. Beaucoup de données
   disponibles (chaque bâtiment fournit des milliers de fenêtres).
2. **Partie B — Extrapolation du total annuel** : à partir des 90 premiers jours observés
   (consommation + météo, au pas journalier), prédire la consommation électrique totale de
   l'année. Peu de données disponibles (une seule observation par bâtiment).

## Données et limite importante

Échantillon : les **100 bâtiments tout-électriques** déjà préparés dans
`timeseries_clustering_multi.ipynb` (`data/raw/timeseries_100_all_electric/`), avec leurs
features physiques .

**Pour la 2 ème partie B , N=100 bâtiments est un échantillon très petit** pour un réseau de neurones
(une seule ligne d'apprentissage par bâtiment, contre des milliers de fenêtres pour la 1 ere partie A).
Les métriques de test sur ~15 bâtiments doivent être lues avec prudence (forte variance). Étendre
l'échantillon de bâtiments tout-électriques serait la première piste d'amélioration si ces
résultats doivent être consolidés.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

ROOT = Path().resolve().parent.parent
DATA_RAW_100 = ROOT / "data" / "raw" / "timeseries_100_all_electric"
DATA_PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "reports" / "figures"

COL = "out.electricity.total.energy_consumption..kwh"
WEATHER_COLS = [
    "out.outdoor_air_drybulb_temp..c",
    "out.outdoor_air_relative_humidity..percentage",
    "out.outdoor_air_wetbulb_temp..c",
    "out.outdoor_humidity_ratio..kgwater_per_kgdryair",
    "out.weather.diffuse_solar_radiation..watt_per_m2",
    "out.weather.direct_normal_solar_radiation..watt_per_m2",
    "out.weather.wind_speed..meter_per_second",
]
SEQ_COLS = [COL] + WEATHER_COLS
N_SEQ_FEATURES = len(SEQ_COLS)

## Chargement des 100 bâtiments : séries horaires + features statiques physiques

In [ ]:
parquet_files = sorted(DATA_RAW_100.glob("*-0.parquet"))
print(f"Bâtiments disponibles : {len(parquet_files)}")

bldg_ids = [int(f.stem.split("-")[0]) for f in parquet_files]

# Features statiques (physiques) alignées sur ces 100 bâtiments, via bldg_id -> position dans X_physical_engineered
meta = pd.read_parquet(DATA_PROCESSED / "metadata_clean.parquet", columns=["bldg_id"])
X_physical = pd.read_parquet(DATA_PROCESSED / "X_physical_engineered.parquet")

pos = meta.reset_index(drop=False).set_index("bldg_id").loc[bldg_ids, "index"].values
static_all = X_physical.iloc[pos].reset_index(drop=True).astype("float32").values
N_STATIC_FEATURES = static_all.shape[1]

print("Features statiques :", static_all.shape)

In [ ]:
# Séries horaires (consommation + météo) pour les 100 bâtiments, pas horaire (8760 h/an)
agg = {COL: "sum", **{c: "mean" for c in WEATHER_COLS}}

hourly_all = []
for f in parquet_files:
    d = pd.read_parquet(f, columns=["timestamp"] + SEQ_COLS)
    d = d.assign(timestamp=lambda x: pd.to_datetime(x["timestamp"]) - pd.Timedelta("15m")).set_index("timestamp")
    d_h = d[SEQ_COLS].resample("h").agg(agg)
    hourly_all.append(d_h[SEQ_COLS].values.astype("float32"))

lengths = {len(h) for h in hourly_all}
assert lengths == {8760}, f"Longueurs incohérentes entre bâtiments : {lengths}"
hourly_all = np.stack(hourly_all, axis=0)  # (100, 8760, n_seq_features)
print("Séries horaires :", hourly_all.shape)

# Séries journalières dérivées (consommation sommée, météo moyennée par jour) — réutilisées
# pour la Partie B, sans relire les fichiers bruts
hourly_reshaped = hourly_all.reshape(100, 365, 24, N_SEQ_FEATURES)
daily_all = np.empty((100, 365, N_SEQ_FEATURES), dtype="float32")
daily_all[:, :, 0] = hourly_reshaped[:, :, :, 0].sum(axis=2)      # consommation : somme journalière
daily_all[:, :, 1:] = hourly_reshaped[:, :, :, 1:].mean(axis=2)   # météo : moyenne journalière
print("Séries journalières :", daily_all.shape)

## Split train/val/test — au niveau bâtiment

Pour les deux partiees, le split se fait **par bâtiment** (70 train / 15 validation / 15 test), pas
par fenêtre temporelle : on teste ainsi la capacité du modèle à généraliser à des bâtiments
jamais vus, en s'appuyant uniquement sur leur fenêtre d'observation et leurs métadonnées
statiques. C'est plus exigeant qu'un split temporel intra-bâtiment, mais plus réaliste pour un
usage en diagnostic énergétique (prédire pour un nouveau logement).

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
perm = rng.permutation(100)
train_idx, val_idx, test_idx = perm[:70], perm[70:85], perm[85:]

print(f"Train : {len(train_idx)} bâtiments")
print(f"Val   : {len(val_idx)} bâtiments")
print(f"Test  : {len(test_idx)} bâtiments")

## Architecture hybride commune (réutilisée pour les deux parties)

`rnn_type` est paramétrable (`"SimpleRNN"`, `"LSTM"`, `"GRU"`) — GRU par défaut (bon compromis
rapidité/performance sur des séquences courtes).

In [ ]:
def build_hybrid_model(seq_len, n_seq_features, n_static_features,
                        rnn_type="GRU", rnn_units=32, static_units=(32, 16),
                        dropout=0.0):
    """Modèle hybride : branche séquentielle (RNN/LSTM/GRU) + branche statique (MLP),
    concaténées puis passées dans une tête de régression commune."""
    rnn_layer = {"SimpleRNN": layers.SimpleRNN, "LSTM": layers.LSTM, "GRU": layers.GRU}[rnn_type]

    seq_input = layers.Input(shape=(seq_len, n_seq_features), name="seq_input")
    x = rnn_layer(rnn_units)(seq_input)
    if dropout > 0:
        x = layers.Dropout(dropout)(x)

    static_input = layers.Input(shape=(n_static_features,), name="static_input")
    s = static_input
    for units in static_units:
        s = layers.Dense(units, activation="relu")(s)
    if dropout > 0:
        s = layers.Dropout(dropout)(s)

    combined = layers.Concatenate()([x, s])
    combined = layers.Dense(max(8, rnn_units // 2), activation="relu")(combined)
    output = layers.Dense(1, name="output")(combined)

    return keras.Model(inputs=[seq_input, static_input], outputs=output, name=f"hybrid_{rnn_type}")

## Partie A — Prévision à court terme (t+1, fenêtre de 24h)

Fenêtre glissante de 24 heures (consommation + météo) → consommation de l'heure suivante.
Chaque bâtiment fournit `8760 - 24 = 8736` fenêtres, donc largement assez de données pour
entraîner un réseau récurrent malgré le faible nombre de bâtiments.

In [ ]:
LOOKBACK = 24

def make_windows_t1(hourly_subset, static_subset):
    """Fenêtres glissantes (lookback, n_seq_features) -> consommation de l'heure suivante,
    avec les features statiques du bâtiment répétées pour chaque fenêtre."""
    n_bldg, T, n_feat = hourly_subset.shape
    n_windows = T - LOOKBACK

    X = np.lib.stride_tricks.sliding_window_view(hourly_subset, LOOKBACK, axis=1)
    X = X[:, :n_windows]
    X = np.moveaxis(X, -1, -2)  # (n_bldg, n_windows, LOOKBACK, n_feat)

    y = hourly_subset[:, LOOKBACK:LOOKBACK + n_windows, 0]  # cible : consommation à t+1

    static_rep = np.repeat(static_subset[:, None, :], n_windows, axis=1)

    return (X.reshape(-1, LOOKBACK, n_feat),
            static_rep.reshape(-1, static_subset.shape[1]),
            y.reshape(-1))


Xa_train, Sa_train, ya_train = make_windows_t1(hourly_all[train_idx], static_all[train_idx])
Xa_val, Sa_val, ya_val = make_windows_t1(hourly_all[val_idx], static_all[val_idx])
Xa_test, Sa_test, ya_test = make_windows_t1(hourly_all[test_idx], static_all[test_idx])

print(f"Train : {Xa_train.shape[0]:,} fenêtres")
print(f"Val   : {Xa_val.shape[0]:,} fenêtres")
print(f"Test  : {Xa_test.shape[0]:,} fenêtres")

In [ ]:
# Normalisation : tous les scalers sont fités UNIQUEMENT sur le train
seq_scaler_a = StandardScaler().fit(Xa_train.reshape(-1, N_SEQ_FEATURES))
Xa_train_sc = seq_scaler_a.transform(Xa_train.reshape(-1, N_SEQ_FEATURES)).reshape(Xa_train.shape).astype("float32")
Xa_val_sc = seq_scaler_a.transform(Xa_val.reshape(-1, N_SEQ_FEATURES)).reshape(Xa_val.shape).astype("float32")
Xa_test_sc = seq_scaler_a.transform(Xa_test.reshape(-1, N_SEQ_FEATURES)).reshape(Xa_test.shape).astype("float32")

static_scaler_a = StandardScaler().fit(Sa_train)
Sa_train_sc = static_scaler_a.transform(Sa_train).astype("float32")
Sa_val_sc = static_scaler_a.transform(Sa_val).astype("float32")
Sa_test_sc = static_scaler_a.transform(Sa_test).astype("float32")

y_scaler_a = StandardScaler().fit(ya_train.reshape(-1, 1))
ya_train_sc = y_scaler_a.transform(ya_train.reshape(-1, 1)).astype("float32").ravel()
ya_val_sc = y_scaler_a.transform(ya_val.reshape(-1, 1)).astype("float32").ravel()

In [ ]:
model_a = build_hybrid_model(LOOKBACK, N_SEQ_FEATURES, N_STATIC_FEATURES, rnn_type="GRU", rnn_units=32)
model_a.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])
model_a.summary()

early_stop_a = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history_a = model_a.fit(
    [Xa_train_sc, Sa_train_sc], ya_train_sc,
    validation_data=([Xa_val_sc, Sa_val_sc], ya_val_sc),
    epochs=15, batch_size=512, callbacks=[early_stop_a], verbose=1,
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_a.history["loss"], label="train")
plt.plot(history_a.history["val_loss"], label="val")
plt.title("Partie A — Courbe d'apprentissage (MSE, cible standardisée)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
pred_test_a_sc = model_a.predict([Xa_test_sc, Sa_test_sc], verbose=0)
pred_test_a = y_scaler_a.inverse_transform(pred_test_a_sc).ravel()

# Baseline de persistance : prédire la consommation de l'heure suivante = dernière heure observée
persistence_pred = Xa_test[:, -1, 0]

def metrics(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"Modèle": label, "RMSE": rmse, "MAE": mae, "R2": r2}

df_metrics_a = pd.DataFrame([
    metrics(ya_test, persistence_pred, "Persistance (t = t-1)"),
    metrics(ya_test, pred_test_a, f"Hybride GRU+MLP ({LOOKBACK}h)"),
]).set_index("Modèle")

df_metrics_a.round(4)

In [ ]:
rng_plot = np.random.default_rng(RANDOM_STATE)
n_points = 3000
sample = rng_plot.choice(len(ya_test), size=min(n_points, len(ya_test)), replace=False)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(ya_test[sample], pred_test_a[sample], alpha=0.25, s=8)
lim = [0, max(ya_test[sample].max(), pred_test_a[sample].max())]
ax.plot(lim, lim, "r--", linewidth=2)
ax.set_xlabel("Consommation réelle à t+1 (kWh)")
ax.set_ylabel("Consommation prédite à t+1 (kWh)")
ax.set_title("Partie A — Réel vs prédit (bâtiments de test, échantillon)")
plt.tight_layout()
plt.savefig(FIGURES / "rnn_mlp_tacheA_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def plot_metrics_bar(df_metrics, title, filename):
    """Barres comparatives RMSE / MAE entre les modèles d'une table de métriques."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    colors = ["#888888", "#1f77b4", "#2ca02c"]
    for ax, metric in zip(axes, ["RMSE", "MAE"]):
        df_metrics[metric].plot(kind="bar", ax=ax, color=colors[: len(df_metrics)])
        ax.set_title(metric)
        ax.set_ylabel(metric)
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=20)
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(FIGURES / filename, dpi=150, bbox_inches="tight")
    plt.show()


plot_metrics_bar(
    df_metrics_a,
    "Partie A — Comparaison des métriques (persistance vs hybride)",
    "rnn_mlp_partieA_metrics_bar.png",
)

In [ ]:
residuals_a = pred_test_a - ya_test

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(residuals_a, bins=80, color="#1f77b4", alpha=0.8)
axes[0].axvline(0, color="r", linestyle="--")
axes[0].set_xlabel("Résidu (prédit − réel, kWh)")
axes[0].set_ylabel("Nombre de fenêtres")
axes[0].set_title("Distribution des résidus")

axes[1].scatter(ya_test[sample], residuals_a[sample], alpha=0.2, s=8)
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set_xlabel("Consommation réelle à t+1 (kWh)")
axes[1].set_ylabel("Résidu (prédit − réel, kWh)")
axes[1].set_title("Résidus vs valeur réelle")

fig.suptitle("Partie A — Analyse des résidus")
plt.tight_layout()
plt.savefig(FIGURES / "rnn_mlp_partieA_residus.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Zoom temporel sur un bâtiment de test : le split par bâtiment fait que les fenêtres
# de chaque bâtiment se suivent dans Xa_test/ya_test/pred_test_a (ordre de test_idx)
n_windows_per_bldg_a = hourly_all.shape[1] - LOOKBACK
bldg_slice = slice(0, n_windows_per_bldg_a)
n_show = 24 * 14  # deux semaines

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ya_test[bldg_slice][:n_show], label="Réel", linewidth=1.2)
ax.plot(pred_test_a[bldg_slice][:n_show], label="Prédit", linewidth=1.2, alpha=0.8)
ax.set_xlabel("Heure")
ax.set_ylabel("Consommation (kWh)")
ax.set_title(f"Partie A — Réel vs prédit dans le temps (bâtiment test #{test_idx[0]}, 2 premières semaines)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "rnn_mlp_partieA_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()

## Partie B — Extrapolation du total annuel (90 premiers jours observés)

À partir des 90 premiers jours de l'année (consommation + météo, au pas **journalier** — la
séquence est donc bien plus courte que pour la Parie A, 90 pas au lieu de 24), on extrapole la
consommation électrique totale de l'année complète.

 Ici, une seule observation par bâtiment (donc 70 exemples d'entraînement seulement) : le
réseau est volontairement gardé petit (peu d'unités, dropout) pour limiter le sur-apprentissage,
mais les résultats restent à interpréter avec prudence — cf. avertissement en introduction.

In [ ]:
K_DAYS = 90

X_seq_annual = daily_all[:, :K_DAYS, :]          # (100, 90, n_seq_features)
y_annual = daily_all[:, :, 0].sum(axis=1)        # consommation totale sur les 365 jours

Xb_train, Xb_val, Xb_test = X_seq_annual[train_idx], X_seq_annual[val_idx], X_seq_annual[test_idx]
Sb_train, Sb_val, Sb_test = static_all[train_idx], static_all[val_idx], static_all[test_idx]
yb_train, yb_val, yb_test = y_annual[train_idx], y_annual[val_idx], y_annual[test_idx]

print(f"Train : {Xb_train.shape[0]} bâtiments  |  Val : {Xb_val.shape[0]}  |  Test : {Xb_test.shape[0]}")

In [ ]:
seq_scaler_b = StandardScaler().fit(Xb_train.reshape(-1, N_SEQ_FEATURES))
Xb_train_sc = seq_scaler_b.transform(Xb_train.reshape(-1, N_SEQ_FEATURES)).reshape(Xb_train.shape).astype("float32")
Xb_val_sc = seq_scaler_b.transform(Xb_val.reshape(-1, N_SEQ_FEATURES)).reshape(Xb_val.shape).astype("float32")
Xb_test_sc = seq_scaler_b.transform(Xb_test.reshape(-1, N_SEQ_FEATURES)).reshape(Xb_test.shape).astype("float32")

static_scaler_b = StandardScaler().fit(Sb_train)
Sb_train_sc = static_scaler_b.transform(Sb_train).astype("float32")
Sb_val_sc = static_scaler_b.transform(Sb_val).astype("float32")
Sb_test_sc = static_scaler_b.transform(Sb_test).astype("float32")

y_scaler_b = StandardScaler().fit(yb_train.reshape(-1, 1))
yb_train_sc = y_scaler_b.transform(yb_train.reshape(-1, 1)).astype("float32").ravel()
yb_val_sc = y_scaler_b.transform(yb_val.reshape(-1, 1)).astype("float32").ravel()

In [ ]:
model_b = build_hybrid_model(
    K_DAYS, N_SEQ_FEATURES, N_STATIC_FEATURES,
    rnn_type="GRU", rnn_units=16, static_units=(16,), dropout=0.3,
)
model_b.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])
model_b.summary()

early_stop_b = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

history_b = model_b.fit(
    [Xb_train_sc, Sb_train_sc], yb_train_sc,
    validation_data=([Xb_val_sc, Sb_val_sc], yb_val_sc),
    epochs=150, batch_size=8, callbacks=[early_stop_b], verbose=0,
)
print(f"Arrêt à l'epoch {len(history_b.history['loss'])} (early stopping)")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_b.history["loss"], label="train")
plt.plot(history_b.history["val_loss"], label="val")
plt.title("Partie B — Courbe d'apprentissage (MSE, cible standardisée)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
pred_test_b_sc = model_b.predict([Xb_test_sc, Sb_test_sc], verbose=0)
pred_test_b = y_scaler_b.inverse_transform(pred_test_b_sc).ravel()

dummy_pred_b = np.full_like(yb_test, yb_train.mean())

df_metrics_b = pd.DataFrame([
    metrics(yb_test, dummy_pred_b, "Dummy (moyenne du train)"),
    metrics(yb_test, pred_test_b, f"Hybride GRU+MLP ({K_DAYS}j observés)"),
]).set_index("Modèle")

df_metrics_b.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(yb_test, pred_test_b, s=60, alpha=0.7)
lim = [0, max(yb_test.max(), pred_test_b.max())]
ax.plot(lim, lim, "r--", linewidth=2)
ax.set_xlabel("Consommation annuelle réelle (kWh)")
ax.set_ylabel("Consommation annuelle prédite (kWh)")
ax.set_title(f"Partie B — Réel vs prédit ({len(yb_test)} bâtiments de test)")
plt.tight_layout()
plt.savefig(FIGURES / "rnn_mlp_tacheB_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plot_metrics_bar(
    df_metrics_b,
    "Partie B — Comparaison des métriques (dummy vs hybride)",
    "rnn_mlp_partieB_metrics_bar.png",
)

In [ ]:
# Réel vs prédit par bâtiment de test, triés par consommation réelle croissante
order = np.argsort(yb_test)
bldg_labels = [str(b) for b in np.array(bldg_ids)[test_idx][order]]

x = np.arange(len(yb_test))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width / 2, yb_test[order], width, label="Réel")
ax.bar(x + width / 2, pred_test_b[order], width, label="Prédit")
ax.set_xticks(x)
ax.set_xticklabels(bldg_labels, rotation=45, ha="right")
ax.set_xlabel("Bâtiment (id)")
ax.set_ylabel("Consommation annuelle (kWh)")
ax.set_title("Partie B — Réel vs prédit par bâtiment (triés par consommation réelle)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "rnn_mlp_partieB_bar.png", dpi=150, bbox_inches="tight")
plt.show()

## Synthèse

- **Architecture hybride** (RNN/LSTM/GRU + MLP) validée sur deux parties de nature différente
  (prévision court terme abondante en données, extrapolation annuelle rare en données), avec le
  même bloc `build_hybrid_model` réutilisé — seul `rnn_type` change pour comparer
  SimpleRNN/LSTM/GRU si besoin.
- **Partie A** : la fenêtre de 24h + météo + métadonnées bat la baseline de persistance — le
  modèle apporte une information réelle au-delà de la simple autocorrélation horaire. Les
  résidus sont globalement centrés sur zéro et le suivi temporel sur un bâtiment de test
  confirme que le modèle capture bien la dynamique journalière.
- **Partie B** : le signal des 90 premiers jours + météo + métadonnées permet d'extrapoler le
  total annuel nettement mieux qu'une baseline naïve, mais **sur seulement 15 bâtiments de
  test** — à confirmer sur un échantillon de bâtiments tout-électriques plus large avant toute
  conclusion définitive.
- **Limite principale** : la Partie B souffre d'un déséquilibre structurel entre le nombre de
  paramètres du réseau et le nombre d'exemples d'entraînement (70). Prochaine étape naturelle :
  étendre l'échantillon tout-électrique au-delà de 100 bâtiments (cf. pipeline de téléchargement
  déjà en place dans `timeseries_clustering_multi.ipynb`) pour fiabiliser ces résultats.